# 02 — CIF Collection & Dataset Organization

**Goal**: Collect CIFs and create four clean datasets:

| Dataset | Description |
|---------|-------------|
| `comp_train.csv` | All training entries — 861 (for composition-only model) |
| `comp_test.csv` | All test entries — 121 (for composition-only model) |
| `struct_train.csv` | Only entries with CIFs — ~339 (for structure-based model) |
| `struct_test.csv` | Only test entries with CIFs — ~69 (for structure-based model) |

**Prerequisites**: 
- Run `01_data_loading.ipynb` first
- OBELiX CIF archives extracted somewhere in `raw_data/`
- `pip install pymatgen`

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
from pymatgen.core import Composition

## Step 1: Collect all CIFs into `cifs/`

In [ ]:
os.makedirs('cifs', exist_ok=True)

# Walk through all of raw_data to find every .cif file (handles nested folders)
copied = 0
for root, dirs, files in os.walk('raw_data'):
    for f in files:
        if f.endswith('.cif'):
            shutil.copy2(os.path.join(root, f), os.path.join('cifs', f))
            copied += 1

cif_count = len([f for f in os.listdir('cifs') if f.endswith('.cif')])
print(f"Copied {copied} CIF files")
print(f"Unique CIFs in cifs/: {cif_count}")

Copied 526 CIF files
Unique CIFs in cifs/: 408


## Step 2: Load clean data and create four datasets

Loads from `clean_train.csv` and `clean_test_DO_NOT_TOUCH.csv` (output of notebook 01).
If those don't exist, re-run `01_data_loading.ipynb` first.

In [ ]:
train = pd.read_csv('clean_train.csv')
test = pd.read_csv('clean_test_DO_NOT_TOUCH.csv')

print(f"Loaded train: {len(train)} rows")
print(f"Loaded test:  {len(test)} rows")

# Mark which entries have CIFs
cif_files = set(f.replace('.cif', '') for f in os.listdir('cifs') if f.endswith('.cif'))
train['has_cif'] = train['id'].isin(cif_files)
test['has_cif'] = test['id'].isin(cif_files)

# Composition-only datasets (everything)
comp_train = train.copy()
comp_test = test.copy()

# Structure-based datasets (only entries with CIFs)
struct_train = train[train['has_cif']].copy()
struct_test = test[test['has_cif']].copy()

print(f"\n{'='*55}")
print(f"FINAL DATASET SUMMARY")
print(f"{'='*55}")
print(f"\n--- Composition-only model ---")
print(f"  comp_train:   {len(comp_train)} entries")
print(f"    OBELiX:     {(comp_train['source']=='obelix').sum()}")
print(f"    Liverpool:  {(comp_train['source']=='liverpool').sum()}")
print(f"  comp_test:    {len(comp_test)} entries")
print(f"\n--- Structure-based model ---")
print(f"  struct_train: {len(struct_train)} entries")
print(f"    OBELiX:     {(struct_train['source']=='obelix').sum()}")
print(f"    Liverpool:  {(struct_train['source']=='liverpool').sum()}")
print(f"  struct_test:  {len(struct_test)} entries")
print(f"\n  CIF files:    {len(cif_files)}")

Loaded train: 861 rows
Loaded test:  121 rows

FINAL DATASET SUMMARY

--- Composition-only model ---
  comp_train:   861 entries
    OBELiX:     478
    Liverpool:  383
  comp_test:    121 entries

--- Structure-based model ---
  struct_train: 339 entries
    OBELiX:     271
    Liverpool:  68
  struct_test:  69 entries

  CIF files:    408


## Step 3: Verify test set integrity

In [ ]:
# 1. No test IDs leaked into train
assert len(set(comp_test['id']) & set(comp_train['id'])) == 0, "DATA LEAK!"
print("✓ No ID overlap between train and test")

# 2. struct datasets are subsets of comp datasets
assert set(struct_train['id']).issubset(set(comp_train['id']))
assert set(struct_test['id']).issubset(set(comp_test['id']))
print("✓ Structure datasets are subsets of composition datasets")

# 3. Every struct entry has a CIF on disk
for entry_id in struct_train['id']:
    assert os.path.exists(f'cifs/{entry_id}.cif'), f"Missing CIF: {entry_id}"
for entry_id in struct_test['id']:
    assert os.path.exists(f'cifs/{entry_id}.cif'), f"Missing CIF: {entry_id}"
print("✓ Every structure entry has a CIF file")

# 4. Test set is OBELiX only
assert (comp_test['source'] == 'obelix').all()
print("✓ Test set is OBELiX only")

print("\nAll checks passed!")

✓ No ID overlap between train and test
✓ Structure datasets are subsets of composition datasets
✓ Every structure entry has a CIF file
✓ Test set is OBELiX only

All checks passed!


In [ ]:
comp_train.to_csv('comp_train.csv', index=False)
comp_test.to_csv('comp_test.csv', index=False)
struct_train.to_csv('struct_train.csv', index=False)
struct_test.to_csv('struct_test.csv', index=False)

print("Saved:")
print(f"  comp_train.csv     ({len(comp_train)} entries)")
print(f"  comp_test.csv      ({len(comp_test)} entries)")
print(f"  struct_train.csv   ({len(struct_train)} entries)")
print(f"  struct_test.csv    ({len(struct_test)} entries)")
print("\n✓ Done! Ready for feature extraction.")

Saved:
  comp_train.csv     (861 entries)
  comp_test.csv      (121 entries)
  struct_train.csv   (339 entries)
  struct_test.csv    (69 entries)

✓ Done! Ready for feature extraction.
